### Dependencies

In [1]:
import openai
import instructor

from pydantic import BaseModel, Field
from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

### RAG Pipeline

In [3]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode= instructor.Mode.RESPONSES_TOOLS
)

In [4]:
class RAGGenerationResponse(BaseModel):
    ans: str = Field(description="Answer to the question")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding


def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

#client.create_with_completion() returns two values
def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )
    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.ans,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

### RAG Pipeline With Grounding Context

In [6]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    ans: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding


def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If you are describing multiple products, list them out as a list.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

#client.create_with_completion() returns two values
def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )
    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.ans,
        'references': answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [9]:
output = rag_pipeline("Do you have any earphones", top_k=10)

In [12]:
output

{'data_object': RAGGenerationResponse(ans='Yes—here are some earphones/earbuds available:\n\n- Volkano Ninja Kids Earphones (3.5mm wired) with carry case and keyring (Red/Blue)\n- Jesebang Wireless Earbuds (Bluetooth 5.2) with charging case, built-in mic (White)\n- 2 Pack iPhone Wired Headphones (3.5mm jack, Apple MFi certified) with microphone & volume control\n- Wekily Bluetooth 5.3 Wireless Earbuds with 40H playtime and charging case (IPX7, running/workout)\n- pamu Wireless Earbuds with ANC/ENC (Bluetooth 5.2) and charging case (Black)\n- Kids Wireless/ Wired Headphones with adjustable headband, Bluetooth 5.0 and 3.5mm jack (Yellow)\n\nIf you tell me whether you want wired or wireless (and your device, e.g., iPhone/android), I can help you pick the best one.', references=[RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids Earphones for Boys with Carry Case and Keyring - 3.5MM Stereo Jack - Wired Earbuds with safe volume limit 85dB (Red/Blue).'), RAGUsedContext(id='B09X9

In [11]:
print(output["answer"])

Yes—here are some earphones/earbuds available:

- Volkano Ninja Kids Earphones (3.5mm wired) with carry case and keyring (Red/Blue)
- Jesebang Wireless Earbuds (Bluetooth 5.2) with charging case, built-in mic (White)
- 2 Pack iPhone Wired Headphones (3.5mm jack, Apple MFi certified) with microphone & volume control
- Wekily Bluetooth 5.3 Wireless Earbuds with 40H playtime and charging case (IPX7, running/workout)
- pamu Wireless Earbuds with ANC/ENC (Bluetooth 5.2) and charging case (Black)
- Kids Wireless/ Wired Headphones with adjustable headband, Bluetooth 5.0 and 3.5mm jack (Yellow)

If you tell me whether you want wired or wireless (and your device, e.g., iPhone/android), I can help you pick the best one.
